# Clase 1 · Fundamentos de la ciencia de datos

**Módulo 1: Introducción y fundamentos estadísticos** · Diplomado en Ciencia de Datos Aplicada · UTFSM

Esta es la **plantilla para trabajar en vivo**: trae la estructura y las instrucciones, y el código se escribe durante la clase. La versión completa y ejecutada queda en el repositorio como referencia.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniopitz/diplomado-cdd/blob/main/01_intro_ecosistema_plantilla_colab.ipynb)

Versión para **Google Colab**: las librerías ya vienen instaladas, no hay nada que configurar.

El hilo de la sesión: comprobar que el entorno funciona y explorar un conjunto de datos real, la Encuesta Origen Destino de Santiago 2012 (EOD), siguiendo la idea de la clase: cada columna de una tabla es una variable de cierto tipo, y el tipo define qué análisis es válido.

## 1. Verificación del entorno

Si esta celda corre sin errores y muestra las versiones, está todo en orden.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("NumPy      ", np.__version__)
print("Pandas     ", pd.__version__)
print("Matplotlib ", matplotlib.__version__)

## 2. NumPy: cálculo sobre arreglos completos

NumPy aporta el arreglo (`array`): una colección de números sobre la que las operaciones se aplican completas, sin escribir ciclos. A eso se le llama vectorización, y es la base de todo el ecosistema: Pandas y Matplotlib trabajan sobre arreglos de NumPy.

Estos cinco valores son duraciones de viaje, en minutos:

In [ ]:
# Cree el arreglo tiempos = np.array([70, 105, 90, 25, 40])
# y calcule su media, su máximo y su equivalente en horas (tiempos / 60)


La división `tiempos / 60` se aplicó a los cinco valores de una vez. Con cinco números da lo mismo; con los cientos de miles de filas de una encuesta, la diferencia de velocidad y de claridad del código es considerable.

## 3. Pandas: abrir una tabla real

Pandas aporta el `DataFrame`: una tabla con columnas etiquetadas, cada una de su propio tipo. Vamos a abrir la tabla de viajes de la EOD 2012 directo desde una URL.

Dos detalles del archivo: los campos van separados por punto y coma (`sep=";"`) y los decimales usan coma (`decimal=","`). El archivo pesa unos 15 MB, así que la descarga puede tomar algunos segundos.

In [ ]:
BASE = (
    "https://raw.githubusercontent.com/daniopitz/cienciadatos/"
    "main/datos/eod_stgo/"
)
# Abra BASE + "viajes.csv" con pd.read_csv (sep=";", decimal=",", low_memory=False)
# y revise las dimensiones con .shape


113.591 filas: una por viaje registrado. Las tres miradas iniciales a cualquier tabla nueva son `shape` (dimensiones), `head` (primeras filas) e `info` (columnas, tipos y valores presentes).

In [ ]:
# Mire las primeras filas de las columnas Hogar, Persona, ComunaOrigen,
# ComunaDestino y TiempoViaje, con head()


In [ ]:
# Revise columnas y tipos con info(max_cols=15)


## 4. Variables cualitativas: contar, no promediar

`ComunaOrigen` guarda números (94, 71, 13...), pero la variable es **nominal**: cada número es un código de comuna. Promediar códigos no significa nada; contar viajes por código, sí. Para variables cualitativas, la herramienta básica es `value_counts`.

In [ ]:
# Cuente los viajes por ComunaOrigen con value_counts y quédese con el top 10


### Traducir códigos: la primera unión de tablas (`merge`)

Los códigos no se leen solos: la encuesta trae tablas de parámetros que los traducen. La de comunas es una tabla chica con dos columnas, `Id` y `Comuna`:

In [ ]:
# Abra BASE + "tablas_parametros/Comunas.csv" (ojo: aquí sep=",") y mire head()


Para ponerle nombre a cada viaje se usa `merge`: toma dos tablas y empareja sus filas por una columna común. Aquí, el `ComunaOrigen` de cada viaje se empareja con el `Id` de la tabla de comunas, y cada fila de viajes queda con su columna `Comuna` a la vista. Es la operación con que se navega cualquier base relacional, y la usaremos durante todo el módulo.

In [ ]:
# Una viajes con comunas: merge(comunas, left_on="ComunaOrigen", right_on="Id")
# Guarde el resultado como viajes_comuna y repita el value_counts, ahora con nombres


El mismo mecanismo, dos veces seguidas, traduce el modo de transporte: `ViajesDifusion.csv` conecta cada viaje con un código de modo, y `ModoDifusion.csv` le pone nombre a ese código.

In [ ]:
modo_por_viaje = pd.read_csv(BASE + "ViajesDifusion.csv", sep=";")
nombres_modo = pd.read_csv(BASE + "tablas_parametros/ModoDifusion.csv", sep=";")
nombres_modo = nombres_modo.rename(columns={"ModoDifusion": "NombreModo"})

# Encadene dos merge: viajes con modo_por_viaje (on="Viaje") y luego con
# nombres_modo (left_on="ModoDifusion", right_on="ID"). Guarde viajes_modo
# y calcule la proporción por NombreModo: value_counts(normalize=True)


La caminata encabeza el reparto, como vimos en la clase. El número no es idéntico al 33,9% de las slides: aquí contamos viajes **de la muestra**, sin aplicar los factores de expansión que convierten la muestra en ciudad. ¿De quién habla cada número? El de la slide, de Santiago; este, de los encuestados. La diferencia entre ambos es el tema de las clases 2 y 3.

Hay un detalle en las categorías: Bip! aparece tres veces (solo, combinado con otro modo público y combinado con otro privado). Para leer el reparto conviene juntarlas en una sola categoría. Esta es una decisión de limpieza, y lo correcto es tomarla de forma explícita en el código, no a mano:

In [ ]:
# Cree la columna ModoAgrupado: donde NombreModo empiece con "Bip!",
# reemplace por "Bip! (solo o combinado)". Pista: .where con .str.startswith
# Luego repita el value_counts(normalize=True)


Una variable cualitativa también se grafica contando. El reparto agrupado, como gráfico de barras:

In [ ]:
reparto = viajes_modo["ModoAgrupado"].value_counts(normalize=True) * 100

# Grafíquelo con ax.barh (invierta el orden con [::-1] para que el mayor
# quede arriba) y rotule el eje x y el título


## 5. Variables cuantitativas: resumir y mirar la forma

`TiempoViaje` sí es un número de verdad: minutos de duración, una variable **continua**. Para las cuantitativas, el resumen básico es `describe`.

In [ ]:
# Resuma TiempoViaje con describe()


La media (36,9 minutos en la muestra) y la mediana (30) no coinciden: hay una cola de viajes largos que arrastra la media hacia arriba. Esa forma se ve mejor en un histograma:

In [ ]:
# Histograma de TiempoViaje: ax.hist con bins=30 y range=(0, 150)
# Recuerde rotular ejes y título


Dos cosas para mirar: la cola larga hacia la derecha (pocos viajes muy largos) y los peaks en los múltiplos de media hora, porque las personas declaran duraciones redondeadas. En la clase 2 pondremos nombre y número a todo esto: media, mediana, dispersión, forma.

Un ejemplo de variable **discreta**, para completar el mapa de la clase: cuántos viajes hizo cada persona en el día. Se obtiene contando filas por persona, y sus valores son 1, 2, 3..., no cualquier decimal.

In [ ]:
# Cuente los viajes de cada persona: groupby(["Hogar", "Persona"]).size()
# y luego la distribución: value_counts().sort_index()


## 6. Cruzar variables: promedios por grupo

Lo más interesante aparece al cruzar una cualitativa con una cuantitativa: por ejemplo, la duración de los viajes según la comuna donde parten. El patrón es `groupby` (partir la tabla en grupos) seguido del resumen que se quiera por grupo, aquí la media.

Usamos `viajes_comuna`, la tabla que ya tiene los nombres puestos:

In [ ]:
# Agrupe viajes_comuna por "Comuna" y calcule la media de TiempoViaje;
# ordene descendente con sort_values y mire el top 10


Y el otro extremo:

In [ ]:
# Las 10 comunas con viajes más cortos en promedio (tail)


Aquí sí tiene sentido promediar: `TiempoViaje` es cuantitativa, y la comuna solo define los grupos. Son medias de la muestra, sin ponderar, así que se leen como descripción de los encuestados, no de la ciudad.

Ya tenemos el kit inicial completo: contar (`value_counts`), traducir códigos (`merge`), resumir (`describe`), mirar la forma (histograma) y comparar grupos (`groupby`).

## 7. Ejercicio

La misma encuesta tiene una tabla de personas, en `BASE + "personas.csv"` (mismo separador y mismo decimal), y la tabla de parámetros `BASE + "tablas_parametros/Sexo.csv"` traduce el código de la columna `Sexo`.

1. Abra la tabla de personas y revise `shape`, `head` e `info`.
2. Clasifique cinco de sus columnas según el esquema de la clase: nominal, ordinal, discreta o continua.
3. Traduzca el código de `Sexo` con su tabla de parámetros usando `merge`, y calcule la proporción de hombres y mujeres con `value_counts(normalize=True)`.
4. Elija una variable cuantitativa, resúmala con `describe` y grafique su histograma.
5. Calcule el promedio de esa variable por grupo de otra columna cualitativa, con `groupby`.
6. Anote una pregunta sobre los viajes de Santiago que le gustaría responder con estas tablas: puede ser el germen de su proyecto final.

In [ ]:
# Su solución


---

**Próxima clase (jueves 27):** estadística descriptiva: qué miden exactamente la media, la mediana y las medidas de dispersión que hoy aparecieron de pasada, y cuándo usar cada una. Traiga el entorno funcionando; si algo falló hoy, revise la sección de problemas frecuentes de la guía de instalación o escríbanos.